# 08.2 Theoretical Value Example

The call is being sold at **$2 below our theoretical value per share covered**. We wish to purchase it and cash in on that difference by delta hedging.

Start with an at-the-money call: stock and strike are both $100, with 12 weeks until expiry. We use 20% volatility and a 5% interest rate. We buy **100 calls, each covering one share**, and short underlying shares to hedge them. Fractional shares are allowed. This is 100 shares of option exposure.

Run the setup and movement-function cells once. Then rerun **Advance one week** as often as you like: each run generates five daily stock moves and adjusts the hedge at the end of the week. After week 12, run **Settle at expiry**. Rerunning setup resets the experiment; change the seed for a different path.

We simulate with the same volatility used to price the call, with no dividends or transaction costs. Interest uses 252 trading days per year. Weekly hedging leaves a replication error, so the final profit will generally differ from the initial $200 pricing advantage plus interest. Stock-sale receipts are cash flows, not automatically profits.

In [16]:
import numpy as np
import pandas as pd
from scipy.stats import norm

initial_stock = 100.0
strike = 100.0
volatility = 0.20
interest_rate = 0.05
stock_drift = interest_rate
trading_days_per_year = 252
expiry_days = 60
number_of_calls = 100
shares_per_call = 1
option_units = number_of_calls * shares_per_call
discount_to_theo = 2.0
seed = 8


def call_value_and_delta(stock, time_left):
    if time_left <= 0:
        payoff = max(stock - strike, 0.0)
        expiry_delta = 1.0 if stock > strike else 0.0
        if stock == strike:
            expiry_delta = 0.5
        return payoff, expiry_delta
    d1 = (
        np.log(stock / strike) + (interest_rate + 0.5 * volatility**2) * time_left
    ) / (volatility * np.sqrt(time_left))
    d2 = d1 - volatility * np.sqrt(time_left)
    value = stock * norm.cdf(d1) - strike * np.exp(
        -interest_rate * time_left
    ) * norm.cdf(d2)
    return float(value), float(norm.cdf(d1))


initial_theo, initial_delta = call_value_and_delta(
    initial_stock, expiry_days / trading_days_per_year
)
purchase_price = initial_theo - discount_to_theo
if purchase_price < max(initial_stock - strike, 0.0):
    raise ValueError(
        "Choose a smaller discount: purchase price is below intrinsic value."
    )

rng = np.random.default_rng(seed)
initial_shares = -option_units * initial_delta
premium_paid = option_units * purchase_price
initial_cash = -premium_paid - initial_shares * initial_stock
initial_edge = option_units * discount_to_theo
state = {
    "day": 0,
    "stock": initial_stock,
    "delta": initial_delta,
    "shares": initial_shares,
    "cash": initial_cash,
    "settled": False,
}
history = []

print(f"ATM stock / strike: ${initial_stock:.2f} / ${strike:.2f}")
print(f"Theoretical call value per share: ${initial_theo:.4f}")
print(f"Offered price, $2 below theo: ${purchase_price:.4f}")
print(
    f"Buy {number_of_calls} calls x {shares_per_call} share(s): pay ${premium_paid:.2f}"
)
print(f"Delta per call share: {initial_delta:.4f}")
print(
    f"Short {-initial_shares:.4f} shares: receive ${-initial_shares * initial_stock:.2f}"
)
print(f"Cash after entry: ${initial_cash:.2f}")
print(f"Initial pricing advantage: ${initial_edge:.2f}")
print("No outside funding: cash includes the premium payment and short-sale proceeds.")

ATM stock / strike: $100.00 / $100.00
Theoretical call value per share: $4.4891
Offered price, $2 below theo: $2.4891
Buy 100 calls x 1 share(s): pay $248.91
Delta per call share: 0.5678
Short 56.7803 shares: receive $5678.03
Cash after entry: $5429.11
Initial pricing advantage: $200.00
No outside funding: cash includes the premium payment and short-sale proceeds.


### Simulate movements in the underlying

Run once to define the simulation. Each weekly run draws the next five daily moves.

In [17]:
def simulate_stock_movements(start_price, number_of_days):
    """Generate fresh daily prices using geometric Brownian motion."""
    dt = 1 / trading_days_per_year
    shocks = rng.standard_normal(number_of_days)
    log_returns = (stock_drift - 0.5 * volatility**2) * dt + volatility * np.sqrt(
        dt
    ) * shocks
    return start_price * np.exp(np.cumsum(log_returns))

### Advance one week

Rerun this cell to advance time and hedge at the new delta. The five daily moves are shown, but we trade only at the week end.

In [30]:
if state["settled"]:
    print("Already settled. Rerun setup to start a new experiment.")
elif state["day"] >= expiry_days:
    print(
        "At expiry. Run the settlement cell below; no more price moves are generated."
    )
else:
    days_this_week = min(5, expiry_days - state["day"])
    daily_prices = simulate_stock_movements(state["stock"], days_this_week)
    day_labels = range(state["day"] + 1, state["day"] + days_this_week + 1)
    print(
        pd.Series(daily_prices, index=day_labels, name="Stock ($)").round(2).to_string()
    )

    old_stock = state["stock"]
    old_delta = state["delta"]
    old_shares = state["shares"]
    old_cash = state["cash"]
    interest = old_cash * np.expm1(
        interest_rate * days_this_week / trading_days_per_year
    )
    state["cash"] += interest
    state["day"] += days_this_week
    state["stock"] = float(daily_prices[-1])
    time_left = (expiry_days - state["day"]) / trading_days_per_year
    call_value, new_delta = call_value_and_delta(state["stock"], time_left)
    state["delta"] = new_delta

    print(
        f"\nDay {state['day']} / {expiry_days}: ${old_stock:.2f} -> ${state['stock']:.2f}"
    )
    print(
        f"Call delta: {old_delta:.4f} -> {new_delta:.4f} ({new_delta - old_delta:+.4f})"
    )
    print(f"Interest earned / charged this week: ${interest:+.2f}")
    print(f"Shares before adjustment: {old_shares:+.4f} (negative means short)")

    shares_to_trade = 0.0
    hedge_cash_flow = 0.0
    if state["day"] < expiry_days:
        target_shares = -option_units * new_delta
        shares_to_trade = target_shares - old_shares
        hedge_cash_flow = -shares_to_trade * state["stock"]
        state["cash"] += hedge_cash_flow
        state["shares"] = target_shares
        action = "BUY back" if shares_to_trade > 0 else "SELL short"
        if abs(shares_to_trade) < 1e-12:
            print("No hedge adjustment needed.")
        else:
            print(
                f"{action} {abs(shares_to_trade):.4f} shares at ${state['stock']:.2f}"
            )
        print(f"Hedge cash flow: ${hedge_cash_flow:+.2f} (positive = received)")
        print(f"Shares after adjustment: {state['shares']:+.4f}")
        print(
            f"Net delta after hedging: {option_units * new_delta + state['shares']:.6f}"
        )
    else:
        print(
            "Expiry reached: no new hedge. Run settlement to collect payoff and close shares."
        )

    portfolio_value = (
        option_units * call_value + state["shares"] * state["stock"] + state["cash"]
    )
    history.append(
        {
            "day": state["day"],
            "stock": state["stock"],
            "delta": new_delta,
            "shares_traded": shares_to_trade,
            "hedge_cash_flow": hedge_cash_flow,
            "interest": interest,
            "cash": state["cash"],
            "shares": state["shares"],
            "portfolio_value": portfolio_value,
        }
    )
    print(f"Cash balance: ${old_cash:.2f} -> ${state['cash']:.2f}")
    print(f"Call theoretical value per share now: ${call_value:.4f}")
    print(f"Combined position value at our theo: ${portfolio_value:.2f}")
    print(
        "Cash balance alone is not profit; it also finances the short-stock obligation."
    )

56    89.18
57    89.92
58    92.85
59    93.79
60    93.66

Day 60 / 60: $89.14 -> $93.66
Call delta: 0.0000 -> 0.0000 (-0.0000)
Interest earned / charged this week: $+0.29
Shares before adjustment: -0.0028 (negative means short)
Expiry reached: no new hedge. Run settlement to collect payoff and close shares.
Cash balance: $290.71 -> $291.00
Call theoretical value per share now: $0.0000
Combined position value at our theo: $290.75
Cash balance alone is not profit; it also finances the short-stock obligation.


### Settle at expiry

After the last week, cash-settle the call and close the short shares. Repeated runs do not settle twice.

In [ ]:
if state["settled"]:
    print(f"Already settled. Final profit: ${state['final_profit']:.2f}")
elif state["day"] < expiry_days:
    print(
        f"{expiry_days - state['day']} trading days remain. Run the weekly cell again first."
    )
else:
    payoff_per_share = max(state["stock"] - strike, 0.0)
    total_payoff = option_units * payoff_per_share
    closing_cash_flow = state["shares"] * state["stock"]
    final_profit = state["cash"] + total_payoff + closing_cash_flow
    growth = np.exp(interest_rate * expiry_days / trading_days_per_year)
    ideal_profit = initial_edge * growth
    present_value_profit = final_profit / growth

    print(f"Expiry stock price: ${state['stock']:.2f}; strike: ${strike:.2f}")
    print(
        f"Call payoff per share: ${payoff_per_share:.2f}; total received: ${total_payoff:.2f}"
    )
    print(f"Cash before settlement: ${state['cash']:.2f}")
    print(
        f"Buy back {-state['shares']:.4f} short shares: pay ${-closing_cash_flow:.2f}"
    )
    print(f"Final profit, all positions closed: ${final_profit:.2f}")
    print(f"Ideal continuous-hedge profit including interest: ${ideal_profit:.2f}")
    print(f"Weekly hedging error: ${final_profit - ideal_profit:+.2f}")
    print(f"Present value of actual profit: ${present_value_profit:.2f}")
    print(f"Initial pricing advantage for comparison (at expiry): ${initial_edge:.2f}")

    state["cash"] = final_profit
    state["shares"] = 0.0
    state["final_profit"] = final_profit
    state["settled"] = True

Expiry stock price: $93.66; strike: $100.00
Call payoff per share: $0.00; total received: $0.00
Cash before settlement: $291.00
Buy back 0.0028 short shares: pay $0.26
Final profit, all positions closed: $290.75
Ideal continuous-hedge profit including interest: $202.40
Weekly hedging error: $+88.35
Present value of actual profit: $287.30
Initial pricing advantage for comparison: $200.00
